# 02. Data preprocessing dan panel grid-triwulan

Input: data mentah beserta manifest. Output: data bersih, panel train/test/test_partial, fitur scaled, dan scaler yang fit hanya pada training.

**Lihat hasil:** [jumlah gempa sebelum/sesudah preprocessing dan contoh data](#s6) | [data kotak-triwulan dan fitur untuk model](#s8). Hasil ada di bawah sel kode pada bagian tersebut.

[Panduan](../00_panduan.ipynb) | [Sebelumnya](../01_data_collection/01_data_collection.ipynb) | [Berikutnya](../03_eda/03_eda.ipynb)

### Daftar bagian

- [1. Inisialisasi dan dependensi](#s1)
- [2. Validasi cakupan collection](#s2)
- [3. Cleaning dan audit data](#s3)
- [4. Proyeksi dan panel lengkap per triwulan](#s4)
- [5. Standardisasi training dan penyimpanan](#s5)
- [6. Jalankan dan lihat data hasil preprocessing](#s6)
- [7. Pemeriksaan kebocoran dan nol kejadian](#s7)
- [8. Lihat data kotak-triwulan dan fitur yang siap dimodelkan](#s8)

<a id="s1"></a>
## 1. Inisialisasi dan dependensi

Jalankan dari atas ke bawah dengan kernel Python 3.12.11 (Seismic Research). Konfigurasi tersentral di config.json.

In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")
candidates = []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "seismic_research_2021_2025"])
ROOT = next((p for p in candidates if (p / "config.json").is_file() and (p / "src/common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Buka notebook dari folder proyek seismic_research_2021_2025.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Image, Markdown, FileLink
from src.common import read_config
cfg = read_config(ROOT)
print("Interpreter:", sys.executable)
print("Python:", sys.version.split()[0])
print("Proyek:", ROOT)
print("Training:", cfg["train_start"], "sampai sebelum", cfg["train_end_exclusive"])
print("Testing sampai sebelum (UTC):", cfg["test_end_exclusive"])

import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
from src.common import ROOT, file_hash, read_config, require, signature, write_json

Interpreter: C:\Users\yudhi\.local\bin\python3.12.exe
Python: 3.12.11
Proyek: C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025
Training: 2021-01-01 sampai sebelum 2026-01-01
Testing sampai sebelum (UTC): 2026-10-26


<a id="s2"></a>
## 2. Validasi cakupan collection

Nol kejadian hanya dapat dibentuk setelah semua interval API berhasil. Manifest diperiksa terhadap rentang tanggal, kesinambungan interval, dan hash CSV.

In [2]:
def validate_collection(root, cfg):
    manifest = json.loads(require(root / "data/raw/collection_manifest.json", "01 collection").read_text())
    if not manifest.get("all_complete") or manifest["raw_sha256"] != file_hash(root / "data/raw/events.csv"):
        raise ValueError("Manifest collection tidak lengkap atau data mentah telah berubah.")
    cursor = pd.Timestamp(cfg["train_start"], tz="UTC")
    for interval in manifest["intervals"]:
        if pd.Timestamp(interval["start"]) != cursor or not interval["complete"]:
            raise ValueError("Ada celah unduhan. Nol kejadian tidak boleh diasumsikan.")
        cursor = pd.Timestamp(interval["end_exclusive"])
    if cursor != pd.Timestamp(cfg["test_end_exclusive"], tz="UTC"):
        raise ValueError("Cakupan unduhan berbeda dari konfigurasi.")
    return manifest

<a id="s3"></a>
## 3. Cleaning dan audit data

UTC dan format waktu campuran diparsing eksplisit. Duplikat ID menyimpan revisi terbaru; ID kosong memakai kunci atribut. Depth di luar 0-800 km ditandai hilang. Magnitudo/depth hilang tidak otomatis diisi nol.

In [3]:
def clean_events(raw, cfg):
    data = raw.copy()
    for col in ["latitude", "longitude", "depth", "magnitude", "geometry_z"]:
        data[col] = pd.to_numeric(data[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    data["time"] = pd.to_datetime(data["time"], errors="coerce", utc=True, format="mixed")
    box = cfg["bbox"]
    valid = (data["time"].notna() & data["latitude"].between(box["min_lat"], box["max_lat"])
             & data["longitude"].between(box["min_lon"], box["max_lon"])
             & data["time"].ge(pd.Timestamp(cfg["train_start"], tz="UTC"))
             & data["time"].lt(pd.Timestamp(cfg["test_end_exclusive"], tz="UTC")))
    valid_type = data["event_type"].isin(cfg["accepted_event_types"])
    rejected_type = data.loc[valid & ~valid_type].copy()
    rejected_type["reason"] = "event_type_not_accepted"
    rejected = data.loc[~valid].copy()
    rejected["reason"] = "invalid_time_coordinate_or_outside_scope"
    data = data.loc[valid & valid_type].copy()
    data["lastupdate"] = pd.to_datetime(data["lastupdate"], errors="coerce", utc=True, format="mixed")
    fallback = data[["time", "latitude", "longitude", "magnitude"]].astype(str).agg("|".join, axis=1)
    id_values = data["eventid"].fillna("").astype(str).str.strip()
    data["dedup_key"] = np.where(id_values.ne(""), "id:" + id_values, "fallback:" + fallback)
    data = data.sort_values("lastupdate", na_position="first")
    duplicates = data.loc[data.duplicated("dedup_key", keep="last")].copy()
    duplicates["reason"] = "duplicate_older_revision"
    data = data.drop_duplicates("dedup_key", keep="last")
    bad_depth = ~data["depth"].between(0, cfg["max_depth_km"])
    bad_magnitude = ~data["magnitude"].between(-2, 10)
    data["depth_invalid"] = bad_depth
    data["magnitude_invalid"] = bad_magnitude
    data.loc[bad_depth, "depth"] = np.nan
    data.loc[bad_magnitude, "magnitude"] = np.nan
    data["year"] = data["time"].dt.year
    data["period"] = data["time"].dt.tz_localize(None).dt.to_period("Q").astype(str)
    data["event_key"] = data["dedup_key"]
    rejected_parts = [frame for frame in [rejected, rejected_type, duplicates] if not frame.empty]
    rejected_all = pd.concat(rejected_parts, ignore_index=True) if rejected_parts else rejected.iloc[:0].copy()
    return data.sort_values("time").reset_index(drop=True), rejected_all

<a id="s4"></a>
## 4. Proyeksi dan panel lengkap per triwulan

Grid utama 50 km memakai LAEA berpusat di Indonesia, dengan origin tetap. Ini 50 km pada bidang proyeksi, bukan jaminan panjang sisi geodesik tepat 50 km. Universe training hanya grid aktif historis. Semua 20 triwulan disertakan. Testing menambahkan grid baru tanpa mengubah training. State NO_EVENTS terpisah dari INCOMPLETE_FEATURES; depth coverage minimal 80%.

In [4]:
def grid_panel(events, cfg, grid_km=None):
    grid_km = grid_km or cfg["grid_km"]
    size = grid_km * 1000
    data = events.loc[events["magnitude"].ge(cfg["min_magnitude"])].copy()
    if data.empty:
        raise ValueError("Tidak ada event di atas ambang magnitudo.")
    forward = Transformer.from_crs("EPSG:4326", cfg["projection"], always_xy=True)
    inverse = Transformer.from_crs(cfg["projection"], "EPSG:4326", always_xy=True)
    x, y = forward.transform(data["longitude"].to_numpy(), data["latitude"].to_numpy())
    data["grid_x"] = np.floor(np.asarray(x) / size).astype(int)
    data["grid_y"] = np.floor(np.asarray(y) / size).astype(int)
    data["grid_id"] = data["grid_x"].astype(str) + "_" + data["grid_y"].astype(str)
    data["shallow"] = np.where(data["depth"].notna(), data["depth"].lt(cfg["shallow_depth_km"]).astype(float), np.nan)
    train_end = pd.Timestamp(cfg["train_end_exclusive"], tz="UTC")
    historic_ids = set(data.loc[data["time"].lt(train_end), "grid_id"])
    if not historic_ids:
        raise ValueError("Tidak ada grid training.")
    all_grids = data[["grid_id", "grid_x", "grid_y"]].drop_duplicates("grid_id")
    aggregate = data.groupby(["grid_id", "period"], observed=True).agg(
        earthquake_count=("event_key", "size"), avg_magnitude=("magnitude", "mean"),
        max_magnitude=("magnitude", "max"), avg_depth=("depth", "mean"),
        depth_count=("depth", "count"), shallow_count=("shallow", "sum")).reset_index()
    start = pd.Period(cfg["train_start"], freq="Q")
    end_ts = pd.Timestamp(cfg["test_end_exclusive"], tz="UTC")
    last = (end_ts - pd.Timedelta(microseconds=1)).tz_localize(None).to_period("Q")
    period_rows = []
    for p in pd.period_range(start, last, freq="Q"):
        a = p.start_time.tz_localize("UTC")
        b = (p + 1).start_time.tz_localize("UTC")
        observed_end = min(b, end_ts)
        complete = observed_end == b
        period_rows.append({"period": str(p), "year": p.year, "quarter": p.quarter,
                            "period_complete": complete, "observed_days": (observed_end-a).total_seconds()/86400,
                            "observed_months": 3.0 if complete else (observed_end-a).total_seconds()/86400/30.4375,
                            "split": "train" if a < train_end else ("test" if complete else "test_partial")})
    panel = all_grids.merge(pd.DataFrame(period_rows), how="cross")
    panel = panel.loc[~panel["split"].eq("train") | panel["grid_id"].isin(historic_ids)].copy()
    panel = panel.merge(aggregate, on=["grid_id", "period"], how="left", validate="one_to_one")
    for col in ["earthquake_count", "depth_count", "shallow_count"]:
        panel[col] = panel[col].fillna(0).astype(int)
    panel["frequency_per_month"] = panel["earthquake_count"] / panel["observed_months"]
    panel["log_frequency"] = np.log1p(panel["frequency_per_month"])
    panel["shallow_fraction"] = panel["shallow_count"] / panel["depth_count"].replace(0, np.nan)
    panel["depth_coverage"] = panel["depth_count"] / panel["earthquake_count"].replace(0, np.nan)
    panel["low_support"] = panel["earthquake_count"].between(1, cfg["low_support_count"]-1)
    panel["new_grid_2026"] = ~panel["grid_id"].isin(historic_ids)
    eligible = panel[cfg["features"]].notna().all(axis=1) & panel["depth_coverage"].ge(0.8)
    panel["state"] = np.select([panel["earthquake_count"].eq(0), eligible], ["NO_EVENTS", "ELIGIBLE"], default="INCOMPLETE_FEATURES")
    lon, lat = inverse.transform((panel["grid_x"]+.5)*size, (panel["grid_y"]+.5)*size)
    panel["center_longitude"], panel["center_latitude"] = lon, lat
    panel["grid_km"] = grid_km
    panel["row_id"] = panel["grid_id"] + "|" + panel["period"]
    panel = panel.sort_values(["period", "grid_id"]).reset_index(drop=True)
    if panel["row_id"].duplicated().any():
        raise ValueError("row_id grid-triwulan duplikat.")
    return panel, data

<a id="s5"></a>
## 5. Standardisasi training dan penyimpanan

Fitur: log1p(frekuensi/bulan), rata-rata M, maksimum M, rata-rata depth, dan proporsi depth <70 km dari event berkedalaman valid. Koordinat, count, density dan frekuensi mentah tidak digandakan sebagai fitur. StandardScaler fit pada baris training ELIGIBLE saja; test hanya transform. Tidak ada target supervised.

In [5]:
def preprocess(root=ROOT):
    root = Path(root)
    cfg = read_config(root)
    validate_collection(root, cfg)
    raw = pd.read_csv(root / "data/raw/events.csv")
    clean, rejected = clean_events(raw, cfg)
    panel, included = grid_panel(clean, cfg)
    training = panel.loc[panel["split"].eq("train") & panel["state"].eq("ELIGIBLE")]
    if len(training) < 10:
        raise ValueError("Kurang dari 10 observasi training yang layak.")
    scaler = StandardScaler().fit(training[cfg["features"]])
    eligible = panel.loc[panel["state"].eq("ELIGIBLE")]
    scaled = pd.DataFrame(scaler.transform(eligible[cfg["features"]]), columns=cfg["features"])
    scaled.insert(0, "row_id", eligible["row_id"].to_numpy())
    output = root / "data/processed"
    clean.to_csv(output / "events_clean.csv", index=False)
    rejected.to_csv(output / "events_rejected.csv", index=False)
    included.to_csv(output / "events_with_grid.csv", index=False)
    panel.to_csv(output / "panel.csv", index=False)
    scaled.to_csv(output / "features_scaled.csv", index=False)
    joblib.dump({"scaler": scaler, "features": cfg["features"], "fit_split": "train", "fit_rows": len(training)}, output / "transform.joblib")
    meta = {"config_sha256": signature(cfg), "raw_sha256": file_hash(root / "data/raw/events.csv"),
            "created_at_utc": pd.Timestamp.now(tz="UTC").isoformat(), "n_raw": len(raw), "n_clean": len(clean),
            "n_rejected": len(rejected), "n_invalid_depth": int(clean["depth_invalid"].sum()),
            "n_train_eligible": len(training), "n_train_periods": int(panel.loc[panel["split"].eq("train"), "period"].nunique()),
            "min_magnitude_provisional": cfg["min_magnitude"], "depth_coverage_min": 0.8,
            "zero_event_policy": "NO_EVENTS terpisah, statistik magnitudo/kedalaman tetap NaN",
            "grid_universe": "train: grid pernah aktif 2021-2025; test: gabungan historis dan grid baru 2026"}
    write_json(output / "metadata.json", meta)
    return panel, clean, meta

<a id="s6"></a>
## 6. Jalankan dan lihat data hasil preprocessing

Lihat jumlah sebelum/sesudah cleaning, jumlah yang memenuhi batas magnitudo, dan contoh kejadian yang digunakan. M >=3.0 adalah ambang eksploratif, belum klaim katalog lengkap. Analisis sensitivitas ambang ada pada notebook 08.

In [6]:
panel, events, metadata = preprocess(ROOT)
jumlah = lambda nilai: f"{int(nilai):,}".replace(",", ".")
included = pd.read_csv(ROOT / "data/processed/events_with_grid.csv")
rejected = pd.read_csv(ROOT / "data/processed/events_rejected.csv")
below_threshold = len(events) - len(included)
display(Markdown(f"""### Hasil pembersihan dan pemilihan data

**{jumlah(len(included))} kejadian gempa memenuhi batas magnitudo minimal {cfg['min_magnitude']:g} dan masuk ke pembentukan grid.**

| Tahap | Jumlah catatan kejadian |
| --- | ---: |
| Masuk dari collection | {jumlah(metadata['n_raw'])} |
| Tidak diteruskan saat cleaning | {jumlah(len(rejected))} |
| Tersisa sesudah cleaning | {jumlah(len(events))} |
| Tidak memenuhi batas magnitudo (termasuk M hilang, jika ada) | {jumlah(below_threshold)} |
| Memenuhi batas magnitudo dan masuk pembentukan grid | {jumlah(len(included))} |

Data yang berada di bawah batas magnitudo tetap ada dalam `events_clean.csv`.
Untuk penelitian utama, `events_with_grid.csv` hanya menyimpan kejadian yang memenuhi ambang tersebut.
**Satu baris kedua file ini masih berarti satu kejadian gempa.**
"""))
assert metadata["n_raw"] == len(events) + len(rejected)
assert len(events) == below_threshold + len(included)
display(Markdown("### Jumlah gempa per tahun sebelum dan sesudah seleksi"))
raw_before = pd.read_csv(ROOT / "data/raw/events.csv")
annual = pd.concat([
    raw_before.assign(year=pd.to_datetime(raw_before["time"], utc=True, format="mixed", errors="coerce").dt.year).groupby("year").size().rename("Dari collection"),
    events.groupby("year").size().rename("Setelah cleaning"),
    included.groupby("year").size().rename("Lolos batas magnitudo")], axis=1).fillna(0).astype(int).rename_axis("Tahun").reset_index()
display(annual)
display(Markdown("### Contoh 10 kejadian yang benar-benar masuk pembentukan grid"))
display(included[["eventid", "time", "latitude", "longitude", "magnitude", "depth", "grid_id", "period"]].head(10).rename(columns={
    "eventid": "ID kejadian", "time": "Waktu UTC", "latitude": "Lintang", "longitude": "Bujur", "magnitude": "Magnitudo",
    "depth": "Kedalaman (km)", "grid_id": "ID kotak", "period": "Triwulan"}))
display(Markdown("### Mengapa ada catatan yang tidak diteruskan?"))
if rejected.empty:
    display(Markdown("Tidak ada catatan yang ditolak saat cleaning."))
else:
    reasons = {"event_type_not_accepted": "Jenis kejadian tidak termasuk jenis gempa yang dipakai",
               "invalid_time_coordinate_or_outside_scope": "Waktu/koordinat tidak valid atau di luar cakupan",
               "duplicate_older_revision": "Duplikat atau revisi lama"}
    display(rejected["reason"].map(lambda value: reasons.get(value, value)).value_counts().rename_axis("Alasan").reset_index(name="Jumlah catatan"))


### Hasil pembersihan dan pemilihan data

**50.198 kejadian gempa memenuhi batas magnitudo minimal 3 dan masuk ke pembentukan grid.**

| Tahap | Jumlah catatan kejadian |
| --- | ---: |
| Masuk dari collection | 78.265 |
| Tidak diteruskan saat cleaning | 52 |
| Tersisa sesudah cleaning | 78.213 |
| Tidak memenuhi batas magnitudo (termasuk M hilang, jika ada) | 28.015 |
| Memenuhi batas magnitudo dan masuk pembentukan grid | 50.198 |

Data yang berada di bawah batas magnitudo tetap ada dalam `events_clean.csv`.
Untuk penelitian utama, `events_with_grid.csv` hanya menyimpan kejadian yang memenuhi ambang tersebut.
**Satu baris kedua file ini masih berarti satu kejadian gempa.**


### Jumlah gempa per tahun sebelum dan sesudah seleksi

,Tahun,Dari collection,Setelah cleaning,Lolos batas magnitudo
0,2021,9148,9148,7189
1,2022,9248,9248,6984
2,2023,10118,10070,7485
3,2024,7188,7184,5601
4,2025,14922,14922,8887
5,2026,27641,27641,14052


### Contoh 10 kejadian yang benar-benar masuk pembentukan grid

,ID kejadian,Waktu UTC,Lintang,Bujur,Magnitudo,Kedalaman (km),ID kotak,Triwulan
0,20210101_0000007,2021-01-01 00:33:29+00:00,-9.02,118.49,4.3,66.0,1_-15,2021Q1
1,20210101_0000028,2021-01-01 02:33:53+00:00,-1.81,120.57,3.9,10.0,5_1,2021Q1
2,20210101_0000036,2021-01-01 03:21:13+00:00,-10.10,120.78,3.1,73.0,6_-17,2021Q1
3,20210101_0000257,2021-01-01 03:44:06+00:00,-3.05,131.89,3.6,34.0,30_-2,2021Q1
4,20210101_0000045,2021-01-01 04:05:16.600000+00:00,-0.19,124.46,4.6,55.0,14_5,2021Q1
5,20210101_0000055,2021-01-01 05:03:10+00:00,-9.18,119.05,3.2,46.0,2_-15,2021Q1
6,20210101_0000061,2021-01-01 05:52:29+00:00,-2.77,130.50,3.3,10.0,27_-1,2021Q1
7,20210101_0000083,2021-01-01 08:44:49+00:00,-9.82,120.68,3.0,18.0,5_-17,2021Q1
8,20210101_0000114,2021-01-01 11:13:22+00:00,-7.58,128.66,4.1,205.0,23_-12,2021Q1
9,20210101_0000113,2021-01-01 11:32:18+00:00,2.13,127.32,3.7,62.0,20_10,2021Q1


### Mengapa ada catatan yang tidak diteruskan?

,Alasan,Jumlah catatan
0,Jenis kejadian tidak termasuk jenis gempa yang...,52


<a id="s7"></a>
## 7. Pemeriksaan kebocoran dan nol kejadian

Statistik M/depth untuk nol event tetap NaN. Setiap grid historis memiliki 20 periode. Tahun 2026 tidak ikut fit scaler.

In [7]:
historical = panel.loc[panel["split"].eq("train")]
assert historical.groupby("grid_id")["period"].nunique().eq(20).all()
empty = panel.loc[panel["state"].eq("NO_EVENTS")]
assert empty[["avg_magnitude","max_magnitude","avg_depth"]].isna().all().all()
assert set(historical["year"]) == set(range(2021,2026))
print("Pemeriksaan panel lulus.")

Pemeriksaan panel lulus.


<a id="s8"></a>
## 8. Lihat data kotak-triwulan dan fitur yang siap dimodelkan

Bagian ini memperlihatkan perbedaan data kejadian gempa, ringkasan kotak-triwulan, dan fitur yang dipakai model. Seluruh CSV dapat dibuka melalui tautan di bawah tabel.

In [8]:
display(Markdown("""### Data kejadian diubah menjadi ringkasan kotak per triwulan

Misalnya, 10 gempa dalam kotak yang sama selama Januari-Maret menjadi **satu baris ringkasan**, dengan jumlah kejadian 10.
Karena itu jumlah baris panel berbeda dari jumlah gempa.

Kotak tanpa kejadian pada suatu triwulan tetap disimpan agar riwayat waktu tidak terputus.
Namun, baris nol kejadian tidak dimasukkan ke model clustering.
"""))
split_names = {"train": "Training 2021-2025", "test": "Testing 2026 (triwulan lengkap)", "test_partial": "Testing 2026 (triwulan belum lengkap)"}
state_names = {"ELIGIBLE": "Ada kejadian dan fitur layak untuk model", "NO_EVENTS": "Nol kejadian pada triwulan ini", "INCOMPLETE_FEATURES": "Ada kejadian, fitur belum lengkap"}
panel_summary = panel.groupby(["split", "state"]).agg(jumlah_baris=("row_id", "size"), jumlah_gempa=("earthquake_count", "sum")).reset_index()
panel_summary["split"] = panel_summary["split"].map(split_names)
panel_summary["state"] = panel_summary["state"].map(state_names)
display(panel_summary.rename(columns={"split": "Bagian data", "state": "Keadaan kotak", "jumlah_baris": "Baris kotak-triwulan", "jumlah_gempa": "Jumlah gempa di dalamnya"}))
assert panel["earthquake_count"].sum() == len(included)
display(Markdown(f"Total panel: **{jumlah(len(panel))} baris kotak-triwulan**, merangkum **{jumlah(len(included))} kejadian gempa**."))
display(Markdown("### Contoh data training yang masuk model (sebelum standardisasi)"))
ready = panel.loc[panel["split"].eq("train") & panel["state"].eq("ELIGIBLE")]
display(ready[["row_id", "earthquake_count", "frequency_per_month", "avg_magnitude", "max_magnitude", "avg_depth", "shallow_fraction"]].head(10).rename(columns={
    "row_id": "Kotak | triwulan", "earthquake_count": "Jumlah gempa", "frequency_per_month": "Gempa per bulan", "avg_magnitude": "Rata-rata M",
    "max_magnitude": "M terbesar", "avg_depth": "Rata-rata kedalaman (km)", "shallow_fraction": "Proporsi dangkal (0-1)"}).round(3))
display(Markdown("### Contoh hasil standardisasi untuk baris yang sama"))
scaled = pd.read_csv(ROOT / "data/processed/features_scaled.csv")
display(scaled.set_index("row_id").loc[ready["row_id"].head(10)].round(3))
display(Markdown("Lima kolom angka di atas dipakai oleh K-Means dan DBSCAN. Nilainya sudah disetarakan skalanya berdasarkan training; angka negatif berarti di bawah rata-rata training, bukan magnitudo atau kedalaman negatif."))
display(Markdown("### Contoh kotak tanpa kejadian"))
display(panel.loc[panel["state"].eq("NO_EVENTS"), ["grid_id", "period", "earthquake_count", "avg_magnitude", "avg_depth", "state"]].head(5))
display(Markdown("`NaN` pada M dan kedalaman berarti tidak ada nilai yang dapat dihitung karena jumlah gempa nol. Angka ini tidak diisi nol dan baris tersebut tidak masuk clustering."))
display(Markdown("### Buka seluruh datanya"))
for filename, label in [
    ("events_clean.csv", "Kejadian setelah cleaning"), ("events_rejected.csv", "Catatan ditolak beserta alasan"),
    ("events_with_grid.csv", "Kejadian lolos batas magnitudo dan ID kotaknya"), ("panel.csv", "Seluruh ringkasan kotak-triwulan"),
    ("features_scaled.csv", "Lima fitur yang sudah disetarakan skalanya")]:
    display(FileLink(str(ROOT / "data/processed" / filename), result_html_prefix=label + ": "))


### Data kejadian diubah menjadi ringkasan kotak per triwulan

Misalnya, 10 gempa dalam kotak yang sama selama Januari-Maret menjadi **satu baris ringkasan**, dengan jumlah kejadian 10.
Karena itu jumlah baris panel berbeda dari jumlah gempa.

Kotak tanpa kejadian pada suatu triwulan tetap disimpan agar riwayat waktu tidak terputus.
Namun, baris nol kejadian tidak dimasukkan ke model clustering.


,Bagian data,Keadaan kotak,Baris kotak-triwulan,Jumlah gempa di dalamnya
0,Testing 2026 (triwulan lengkap),Ada kejadian dan fitur layak untuk model,2625,13385
1,Testing 2026 (triwulan lengkap),Nol kejadian pada triwulan ini,2892,0
2,Testing 2026 (triwulan belum lengkap),Ada kejadian dan fitur layak untuk model,298,667
3,Testing 2026 (triwulan belum lengkap),Nol kejadian pada triwulan ini,1541,0
4,Training 2021-2025,Ada kejadian dan fitur layak untuk model,13471,36146
5,Training 2021-2025,Nol kejadian pada triwulan ini,21689,0


Total panel: **42.516 baris kotak-triwulan**, merangkum **50.198 kejadian gempa**.

### Contoh data training yang masuk model (sebelum standardisasi)

,Kotak | triwulan,Jumlah gempa,Gempa per bulan,Rata-rata M,M terbesar,Rata-rata kedalaman (km),Proporsi dangkal (0-1)
4,-10_-15|2021Q1,4,1.333,3.500,3.8,14.00,1.00
5,-10_-16|2021Q1,2,0.667,3.300,3.5,10.00,1.00
6,-10_-17|2021Q1,1,0.333,3.100,3.1,10.00,1.00
7,-10_-18|2021Q1,1,0.333,4.000,4.0,10.00,1.00
8,-10_-19|2021Q1,1,0.333,4.400,4.4,48.00,1.00
14,-11_-14|2021Q1,1,0.333,3.100,3.1,111.00,0.00
15,-11_-15|2021Q1,3,1.000,3.700,4.3,27.00,1.00
16,-11_-16|2021Q1,2,0.667,3.600,4.0,10.00,1.00
29,-12_-15|2021Q1,4,1.333,3.300,3.8,36.25,0.75
30,-12_-16|2021Q1,4,1.333,3.575,3.7,10.00,1.00


...

C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025\data\processed\events_clean.csv

C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025\data\processed\events_rejected.csv

C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025\data\processed\events_with_grid.csv

C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025\data\processed\panel.csv

C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025\data\processed\features_scaled.csv

## Referensi dan batas interpretasi

[Dokumentasi EMSC](https://www.seismicportal.eu/fdsn-wsevent.html), [clustering scikit-learn](https://scikit-learn.org/stable/modules/clustering.html), dan docs/metodologi.md. Hasil adalah profil aktivitas katalog dalam bounding box dan periode yang dipilih. Interpretasi risiko, prediksi kejadian gempa, serta klaim perubahan fisik memerlukan bukti tambahan.